## Analysing SegFormer segmentation model
---

In this notebook, we are going to fine-tune SegFormerForSemanticSegmentation on a custom semantic segmentation dataset. In semantic segmentation, the goal for the model is to label each pixel of an image with one of a list of predefined classes.

## Imports 
---

In [ ]:
#external
from tqdm.notebook import trange, tqdm
import pandas as pd
from dataclasses import dataclass
import matplotlib.pyplot as plt

#model
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

#metrics
import evaluate
import torch.nn.functional as F

#utils
from src.utils.dataset import load_foodseg103, decode_image_from_bytes
from src.utils.visualization import predict_random_images

#constants
from src.constants.category_id import CATEGORY_ID

In [ ]:
torch.cuda.empty_cache()

## Testing on custom dataset
---

In [ ]:
SEED = 42
IMAGE_SIZE = 512
LEARNING_RATE = 0.0001
BATCH_SIZE = 8
NUM_EPOCHS = 1   
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "nvidia/segformer-b1-finetuned-ade-512-512"

### Defining Dataset

In [ ]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, image_dataset_df:pd.DataFrame, image_processor:SegformerImageProcessor):
        self.image_dataset = image_dataset_df.reset_index(drop=True)
        self.image_processor = image_processor

    def __len__(self):
        return self.image_dataset.shape[0]
    
    def __getitem__(self, index):
        image_information = self.image_dataset.loc[index]
        image_decoded = decode_image_from_bytes(image_information["image"])
        mask = decode_image_from_bytes(image_information["label"])
        encoded_inputs = self.image_processor(image_decoded, segmentation_maps=mask, return_tensors="pt")
        for k,v in encoded_inputs.items():
          encoded_inputs[k].squeeze_() # remove batch dimension
        return encoded_inputs

In [ ]:
image_processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
image_processor.do_reduce_labels = False
image_processor.size = IMAGE_SIZE

In [ ]:
df = load_foodseg103()
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
test_df, val_df = train_test_split(temp_df, test_size=0.15, random_state=SEED)

In [ ]:
train_dataset = SemanticSegmentationFoodDataset(train_df, image_processor)
test_dataset = SemanticSegmentationFoodDataset(test_df, image_processor)
val_dataset = SemanticSegmentationFoodDataset(val_df, image_processor)
print(f"Number of training examples: {train_dataset.__len__()}")
print(f"Number of testing examples: {test_dataset.__len__()}")
print(f"Number of validation examples: {val_dataset.__len__()}")

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

### Model definition

In [ ]:
food_model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORY_ID),
    id2label=CATEGORY_ID,
    label2id={v: k for k, v in CATEGORY_ID.items()},
    ignore_mismatched_sizes=True
)
for param in food_model.base_model.parameters():
    param.requires_grad = False
food_model.to(DEVICE);

### Model training

In [ ]:
optimizer = torch.optim.AdamW(food_model.parameters(), lr=LEARNING_RATE)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_metric = None
        self.no_improvement_count = 0
    
    def check_early_stop(self, val_metric_value):
        if self.best_metric is None or val_metric_value >= self.best_metric:
            self.best_metric = val_metric_value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                return True
        return False

In [ ]:
@dataclass
class TrainerConfiguration:
    model: SegformerForSemanticSegmentation
    optimizer: torch.optim
    train_dataloader: DataLoader
    val_dataloader: DataLoader
    device: str
    category_id: list
    output_dir: str = "src/model"
    num_epochs: int = 10

class SegmentationTrainer:
    def __init__(self, config: TrainerConfiguration, early_stopping: EarlyStopping):
        self.config = config
        self.early_stopping = early_stopping
        self.metric_history = {"train_loss":[], "val_loss": [], "mean_iou": []}

    def train(self):
        epoch_iterator = tqdm(range(NUM_EPOCHS), desc="Epochs", position=0, leave=True)
        for epoch in epoch_iterator:
            self.config.model.train()
            epoch_loss = 0.0
            batch_iterator = tqdm(self.config.train_dataloader, desc=f"Epoch {epoch+1}", position=1, leave=False)
            for batch in batch_iterator:
                pixel_values = batch["pixel_values"].to(self.config.device)
                labels = batch["labels"].to(self.config.device)
                self.config.optimizer.zero_grad()
                outputs = self.config.model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
                loss.backward()
                self.config.optimizer.step()
                epoch_loss += loss.item()
                batch_iterator.set_postfix(batch_loss=loss.item())
            avg_loss = epoch_loss / len(self.config.train_dataloader)
            val_loss, mean_iou = self.validation()
            epoch_iterator.set_postfix(train_loss=avg_loss, val_loss=val_loss, mean_iou=mean_iou)
            self._add_metrics_history(avg_loss, val_loss, mean_iou)
            self.config.model.save_pretrained(f"{self.config.output_dir}/epoch_{epoch+1}")
            if self.early_stopping.check_early_stop(mean_iou):
                print("No improvements were found, using early stopping...")
                break

    def validation(self):
        self.config.model.eval()
        total_loss = 0.0
        val_mean_iou = evaluate.load("mean_iou")
        with torch.no_grad():
            for batch in tqdm(self.config.val_dataloader, desc="Validating...", position=2, leave=False):
                pixel_values = batch["pixel_values"].to(self.config.device)
                labels = batch["labels"].to(self.config.device)
                outputs = self.config.model(pixel_values=pixel_values, labels=labels)
                loss, logits = outputs.loss, outputs.logits
                upsampled_logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
                total_loss += loss.item()
                predicted = upsampled_logits.argmax(dim=1).detach().cpu().numpy()
                labels = labels.detach().cpu().numpy()
                val_mean_iou.add_batch(predictions=predicted, references=labels)
        avg_loss = total_loss / len(self.config.val_dataloader)
        metrics = val_mean_iou.compute(num_labels=len(self.config.category_id), reduce_labels=False, ignore_index=255)
        return avg_loss, metrics["mean_iou"]

    def plot_metrics(self):
        plt.figure(figsize=(12, 6))
        plt.plot(self.metric_history["train_loss"], label="Train Loss")
        plt.plot(self.metric_history["val_loss"], label="Validation Loss")
        plt.plot(self.metric_history["mean_iou"], label="Mean IoU")
        plt.xlabel("Epochs")
        plt.ylabel("Metrics")
        plt.title("Training and Validation Metrics")
        plt.legend()
        plt.show()

    def _add_metrics_history(self, train_loss, val_loss, mean_iou):
        self.metric_history["train_loss"].append(train_loss)
        self.metric_history["val_loss"].append(val_loss)
        self.metric_history["mean_iou"].append(mean_iou)

In [ ]:
trainer_config = TrainerConfiguration(
    model=food_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,
    val_dataloader=validation_dataloader,
    device=DEVICE,
    category_id=CATEGORY_ID,
    output_dir="src/model",
    num_epochs=NUM_EPOCHS,
)

trainer = SegmentationTrainer(config=trainer_config, early_stopping=EarlyStopping(patience=8))
trainer.train()

### Testing in new image

In [ ]:
trainer.plot_metrics()

In [ ]:
food_model.eval();

In [ ]:
predict_random_images(food_model, image_processor, test_dataset, num_images=10)

## References
[1] https://github.com/NielsRogge/Transformers-Tutorials/tree/master/SegFormer

[2] https://blog.roboflow.com/how-to-train-segformer-on-a-custom-dataset-with-pytorch-lightning/#create-a-dataset